## Legal Expert Knowledge Worker

This is a question-answering assistant with access to the Constitution (supreme law), statutory law, English common law, customary law and Islamic law documents. This will make it an expert in legal problems and it will answer any legal questions thrown to it.

### Note: The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

### PART A: Retrive the documents and Divide them into chunks

The documents will be retrived from an external source. e.g. Google Drive

In [8]:
%pip install google-api-python-client google-auth google-auth-oauthlib
%pip install 'markitdown[all]'


Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 2.1 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 1.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 2.0 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 2.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 2.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 1.9 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [azure-identity]m [azure-identity]tintelligence]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

import tiktoken
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

from google_drive_client import GoogleDriveClient

google_drive_credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
google_drive_folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')

print(f"Credentials path: {google_drive_credentials_path}")
print(f"Google Drive Folder ID: {google_drive_folder_id}")

# Google Drive client:
collector = GoogleDriveClient(google_drive_credentials_path, google_drive_folder_id)
collector.collect_files()

# Files from a Google Drive folder in markdown format.
documents = collector.get_collection()

# Entire knowledge base
entire_knowledge_base = collector.entire_knowledge_base

print(f"Found {len(documents)} files in the knowledge base")

Credentials path: /home/thomas/Downloads/googleUserContent.json
Google Drive Folder ID: 1dOFomY-wQD0NmBDA1XAH8eUqLaE2A0iz
Total characters in knowledge base: 8,339,952
Found 2 files in the knowledge base


In [2]:

from openai import OpenAI

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

In [ ]:
# How many tokens in all the documents?
model = MODEL_MAP["GPT"]["model"]
encoding = tiktoken.encoding_for_model(model)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {model}: {token_count:,}")

print(documents['The_Constitution_of_Kenya_2010.md'][0:1000])

Total tokens for gpt-4o-mini: 8,622,905


KeyError: 'The_Constitution_of_Kenya_2010'

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = text_splitter.split_documents(documents)

# print(f"Divided into {len(chunks)} chunks")
# print(f"First chunk:\n\n{chunks[0]}")

# chunks[100]

AttributeError: 'str' object has no attribute 'page_content'